# shop

> A trolley the agent can fill: `fossick.shop` behind a small interface, and the weekly
> grocery run it was written for.

In [ ]:
#| default_exp shop

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
import tempfile
from pathlib import Path
from fastcore.test import test_eq, test_fail

In [ ]:
#| export
import json
from fastcore.basics import store_attr
from ramabana.core import AgentError, agent_err
from ramabana.tools import clip, err

## A cart, as an interface

`Host` is the harness's one dependency on the world, and a shopping session does not belong in it: an agent editing a repo has no business holding a trolley, and most hosts have no browser to hold one with. A cart is an *extension*. Registered through `Registry.tool`, dropped into a config directory, absent unless someone asked for it.

`Cart` is the interface, for the same reason `Host` is one. `FossickCart` drives a real logged-in Chrome. `FakeCart` is an in-memory double, which is what the tests and the worked example below run against. Filling a real trolley is not something a doc build should do.

In [ ]:
#| export
MAX_PRODUCTS = 24        # products listed back per search. A supermarket page holds far more
SHOP_PORT = 9223         # fossick's persistent debug Chrome
SHOP_TOUT = 180


class CartError(AgentError):
    "A cart action could not be carried out: no such product, no add control, nothing matched."
    pass


class Cart:
    """One shopping session: find things, put them in, read the trolley back.

    Every method may raise, and the tools in `cart_tools` catch and report rather than let an
    exception end a turn. The same contract `Host` has. The trolley is the source of truth:
    `add` is expected to *verify* that the cart moved rather than trust that a click worked,
    because an agent that believes it bought milk is worse than one that says it is not sure.
    """

    def open(self, url):
        "Point the session at a store. Returns the url actually landed on."
        raise NotImplementedError

    def find(self, query, limit=MAX_PRODUCTS):
        "Search the current store. Returns `[{i, title, price, url}]`. `i` is what `add` takes."
        raise NotImplementedError

    def add(self, item, qty=1, variant=None):
        "Add `item` (an `i` from `find`, or a title) and verify it landed. Returns `{ok, item, ...}`."
        raise NotImplementedError

    def lines(self):
        "What is in the trolley now, one row per line."
        raise NotImplementedError

    def total(self):
        "`{count, subtotal}` for the whole trolley."
        raise NotImplementedError

    @property
    def where(self):
        "The url the session is currently on."
        raise NotImplementedError

## The real one

`FossickCart` drives a logged-in Chrome tab through `fossick.shop`.

`fossick.shop` already does the hard part. Injecting its helpers into the page, finding the add control, posting to `/cart/add.js` on Shopify, and watching the cart for a signal that something actually changed. `FossickCart` is the adapter, and it is thin on purpose.

Both stores in the example are already in fossick's `SITES` table. Their search and trolley urls are known rather than guessed. Coles is bot-walled from datacentre IPs and Ceres is a members' shop, which is why this drives a *logged-in* Chrome rather than fetching anything.

In [ ]:
#| export
class FossickCart(Cart):
    "Real Chrome trolley via `fossick.shop`. Session opens on first use and is reused."

    def __init__(self, port=SHOP_PORT, tout=SHOP_TOUT, headless=None):
        store_attr()
        self._s = None

    def _shop(self, url=None):
        from fossick.shop import shop
        if self._s is None:
            self._s = shop(url, port=self.port, headless=self.headless, tout=self.tout, resume=url is None)
        elif url: self._s.open(url)
        return self._s

    @property
    def where(self): return self._shop().url

    def open(self, url): return self._shop(str(url)).url

    def find(self, query, limit=MAX_PRODUCTS): return self._shop().search(str(query), int(limit))

    def add(self, item, qty=1, variant=None):
        return self._shop().add(item, qty=int(qty), variant=variant or None)

    def lines(self): return self._shop().lines()

    def total(self): return self._shop().cart()

    def remove(self, line): return self._shop().remove(line)

    def stores(self):
        "The stores fossick knows the search and trolley urls for."
        from fossick.shop import SITES
        return {k: v.get('note', '') for k, v in SITES.items()}

## The double

`FakeCart` records adds for tests without opening a browser.

`FakeCart` is a shop with a fixed catalogue and a dict for a trolley. It matches products the way fossick does. By index, then exact title, then substring. It raises with the real options rather than guessing, because "no product matching 'milk'" plus a list of titles is a thing a model can recover from and a silent wrong add is not.

In [ ]:
#| export
#: Two stores, because the interesting case is a run that spans both: a supermarket for the
#: staples and a co-op box scheme for the fruit.
CATALOGUE = {
    'coles.com.au': [('Full Cream Milk 2L', 3.60), ('Sourdough Loaf 680g', 6.00),
                     ('Free Range Eggs 12pk', 8.50), ('Baby Spinach 120g', 4.00),
                     ('Extra Virgin Olive Oil 750ml', 12.00), ('Bananas Cavendish 1kg', 4.90)],
    'ceresfairfood.org.au': [('Seasonal Fruit Box - Medium', 39.00), ('Seasonal Veg Box - Medium', 45.00),
                             ('Organic Lemons 500g', 6.50)],
}


class FakeCart(Cart):
    "A cart with a fixed catalogue and no browser: what the tests and the docs shop at."

    def __init__(self, catalogue=None, url='https://www.coles.com.au'):
        self.catalogue = dict(catalogue or CATALOGUE)
        self._url, self._trolley, self._found = url, [], []

    @property
    def where(self): return self._url

    def _store(self):
        return next((v for k, v in self.catalogue.items() if k in self._url), [])

    def open(self, url):
        self._url, self._found = str(url), []
        return self._url

    def _rows(self, query=''):
        q = str(query).lower()
        return [dict(i=i, title=t, price=f'${p:.2f}', url=f'{self._url}/p/{i}')
                for i, (t, p) in enumerate(tp for tp in self._store() if q in tp[0].lower())]

    def find(self, query, limit=MAX_PRODUCTS):
        self._found = self._rows(query)[:int(limit)]
        return self._found

    def _match(self, want):
        """Index against the last search. A title against the whole store.

        The real one re-reads the products on the page for every add. A title that is not in
        the last search still resolves as long as the store stocks it. Which is what lets an
        agent add by the exact title it just read back. An index cannot work that way: it only
        means anything relative to the search that produced it.
        """
        page = self._found or self._rows()
        if isinstance(want, int) or str(want).isdigit():
            hit = next((p for p in page if p['i'] == int(want)), None)
            if hit is None: raise CartError(f'no product #{want}; the page has {len(page)}')
            return hit
        w = str(want).lower()
        for pool in (page, self._rows()):
            for test in (lambda t: w == t.lower(), lambda t: w in t.lower()):
                if (hits := [p for p in pool if test(p['title'])]):
                    return min(hits, key=lambda p: len(p['title']))
        raise CartError(f'no product matching {want!r} at {self._url}. '
                        f'Stocked: {[p["title"] for p in self._rows()]}')

    def add(self, item, qty=1, variant=None):
        it, qty = self._match(item), int(qty)
        before = self.total()
        line = next((l for l in self._trolley if l['title'] == it['title']), None)
        if line: line['qty'] += qty
        else: self._trolley.append(dict(i=len(self._trolley), title=it['title'], price=it['price'], qty=qty))
        after = self.total()
        return dict(ok=after != before, item=it, qty=qty, before=before, after=after)

    def lines(self): return list(self._trolley)

    def total(self):
        n = sum(l['qty'] for l in self._trolley)
        sub = sum(float(l['price'].lstrip('$')) * l['qty'] for l in self._trolley)
        return dict(count=n, subtotal=f'${sub:.2f}')

    def remove(self, line):
        it = self._match(line)
        self._trolley = [l for l in self._trolley if l['title'] != it['title']]
        return dict(ok=True, removed=it['title'], after=self.total())

    def stores(self): return {k: 'fake catalogue' for k in self.catalogue}

In [ ]:
cart = FakeCart()
cart.find('milk')

In [ ]:
cart.add('milk', qty=2)
cart.add('Sourdough Loaf 680g')
cart.total()

Matching failures name the options instead of guessing, which is the difference between a model that recovers and a model that buys the wrong thing.

In [ ]:
test_fail(lambda: cart.add('caviar'), contains='no product matching')
test_fail(lambda: cart.add(99), contains='no product #99')

## The tools

Host methods that expose the trolley to the model.

In [ ]:
#| export
def cart_tools(cart):
    "The trolley, as tools. `cart_add` and `cart_remove` are in `WRITE_TOOLS`: they spend money."

    def cart_stores() -> str:
        "The stores whose search and trolley pages are known, and anything worth knowing about each."
        try: return clip('\n'.join(f'{k}  {v}' for k, v in cart.stores().items()))
        except Exception as e: return err('could not list stores', e)

    def cart_open(url: str) -> str:
        "Point the shopping session at a store, or at one product page. Do this before searching."
        try: return f'now at {cart.open(url)}'
        except Exception as e: return err(f'could not open {url}', e)

    def cart_find(query: str, limit: int = 10) -> str:
        """Search the store you are on. Returns numbered products. The number is what `cart_add` takes.

        Search one item at a time and read the titles back before adding. Supermarket search
        is fuzzy, and 'milk' matches oat milk, condensed milk and a milk frother.
        """
        try:
            rows = cart.find(query, int(limit))
            if not rows: return f'no products matching {query!r} at {cart.where}'
            return clip('\n'.join(f"{r['i']:>3}  {r['price'] or '?':>8}  {r['title']}" for r in rows))
        except Exception as e: return err('search failed', e)

    def cart_add(item: str, qty: int = 1, variant: str = '') -> str:
        """Put a product in the trolley. `item` is a number from `cart_find`, or an exact title.

        The result says whether the trolley actually moved. `ok=false` means it did not and the
        item is NOT in the cart. `ok=null` means the site gave no signal to check against. Confirm with `cart_show` before telling the user it is done.
        """
        try:
            r = cart.add(item, qty=int(qty), variant=variant or None)
            ok = r.get('ok')
            state = 'added' if ok else ('UNVERIFIED' if ok is None else 'DID NOT ADD')
            title = (r.get('item') or {}).get('title', item)
            return clip(f"{state}: {qty} x {title}\ntrolley now {json.dumps(r.get('after'))}"
                        + (f"\n{r.get('error')}" if r.get('error') else ''))
        except Exception as e: return err(f'could not add {item!r}', e)

    def cart_show() -> str:
        "Read the trolley back: every line, and the count and subtotal."
        try:
            ls, t = cart.lines(), cart.total()
            body = '\n'.join(f"{l.get('qty', 1)} x {l.get('title')}  {l.get('price', '')}" for l in ls)
            return clip(f"{t.get('count')} items, subtotal {t.get('subtotal')}\n{body}")
        except Exception as e: return err('could not read the trolley', e)

    def cart_remove(line: str) -> str:
        "Take one line out of the trolley, by its number in `cart_show` or by exact title."
        try: return clip(json.dumps(cart.remove(line), default=str))
        except Exception as e: return err(f'could not remove {line!r}', e)

    return [cart_stores, cart_open, cart_find, cart_add, cart_show, cart_remove]

Registering them is three lines in a config directory, and nothing in the harness changes:

```python
# ~/.config/ramabana/extensions/shopping.py
from ramabana.shop import FossickCart, cart_tools

def setup(reg):
    for t in cart_tools(FossickCart()): reg.tool(t)
```

In [ ]:
from ramabana.tools import Registry, WRITE_TOOLS
reg = Registry()
for t in cart_tools(FakeCart()): reg.tool(t)
[t.__name__ for t in reg.tools]

`cart_add` and `cart_remove` are in `WRITE_TOOLS`. An `Approvals` policy gates them the same way it gates writing a file. Spending money is at least as worth a confirmation.

In [ ]:
test_eq({'cart_add', 'cart_remove'} <= WRITE_TOOLS, True)

## The weekly shop

A scripted multi-store run used as an end-to-end example.

The whole thing end to end: a reminder set once, polled later, and acted on.

Nothing here is mocked except the model's decisions and the trolley. The vault is real, the watch is real, the poll is real, and every tool call below runs the actual tool. `ScriptedBackend` chooses *which* tool to call and streams the words, and the harness does the rest.

In [ ]:
import time
from ramabana import Agent
from ramabana.vault import VaultHost
from ramabana.testing import ScriptedBackend, Step

tmp = Path(tempfile.mkdtemp())
host = VaultHost(roots=[tmp], vault=tmp/'vault.db', index=False, web=False)
host.open_vault(wait=True)

**Once, weeks ago.** The list is written down, and a standing reminder is set against it. `start` is when it first comes due. Backdated here so that the rest of the page is a session happening on the morning it fires, rather than a simulated clock.

In [ ]:
host.remember('Weekly Coles order: milk, sourdough, eggs, baby spinach, olive oil. '
              'Plus a seasonal fruit box from Ceres Fair Food.',
              title='the weekly shop', tags=['groceries'])

due = host.watch('Put the weekly groceries in the Coles trolley, and add a Ceres fruit box.',
                 action='remind', every='1w', note='weekly shop', start=time.time() - 1)
int(due['every'])

Nobody has polled yet. As far as the vault is concerned this is simply outstanding.

In [ ]:
test_eq(len(host.watches(due_only=True)), 1)

**The morning it fires.** The agent gets the host's own tools plus the cart's, and its first move is to poll: nothing else in the system knows the reminder came due. `gpt-mini` is `openai/gpt-5.6-luna`. Fast and cheap, which is what a shopping run wants, being a great many small verifiable steps rather than one hard inference.

In [ ]:
shop = FakeCart()
agent = Agent(host, model='gpt-mini', extensions=False)
tools = {t.__name__: t for t in agent.tools} | {t.__name__: t for t in cart_tools(shop)}
agent.routing.turn

In [ ]:
script = [
    Step('Checking what is due.', tool=('poll_watches', {})),
    Step('There is a weekly shop. Pulling the list.',
         tool=('memory_search', {'query': 'weekly Coles order list', 'limit': 2})),
    Step('Opening Coles.', tool=('cart_open', {'url': 'https://www.coles.com.au'})),
    *[s for item in ['milk', 'sourdough', 'eggs', 'baby spinach', 'olive oil']
      for s in (Step(f'Looking for {item}.', tool=('cart_find', {'query': item})),
                Step(f'Adding {item}.', tool=('cart_add', {'item': item})))],
    Step('Now the fruit box.', tool=('cart_open', {'url': 'https://members.ceresfairfood.org.au'})),
    Step('', tool=('cart_find', {'query': 'fruit box'})),
    Step('', tool=('cart_add', {'item': 'Seasonal Fruit Box - Medium'})),
    Step('Reading the trolley back.', tool=('cart_show', {})),
    Step('Done: five staples in the Coles trolley and a medium seasonal fruit box from Ceres.'),
]

be = ScriptedBackend(steps=script, token_delay=0)
be.refresh('', list(tools.values()))
print(be.send('anything outstanding this week?'))

The reminder fired inside the turn, not before it: `poll_watches` was the agent's own first tool call, and the note it filed is what `memory_search` found on the second.

In [ ]:
[h['content'] for h in be.hist if h['role'] == 'tool'][0]

The trolley is the thing to check, not the transcript. `cart_add` verified each one against the cart rather than against the click.

In [ ]:
shop.total()

In [ ]:
#| hide
test_eq(shop.total()['count'], 6)
test_eq({l['title'] for l in shop.lines()},
        {'Full Cream Milk 2L', 'Sourdough Loaf 680g', 'Free Range Eggs 12pk',
         'Baby Spinach 120g', 'Extra Virgin Olive Oil 750ml', 'Seasonal Fruit Box - Medium'})

The reminder rescheduled itself. Next week this happens again without anyone setting it up again.

In [ ]:
w = host.watches()[0]
test_eq(w['runs'], 1)
test_eq(w['last_status'], 'ok')
test_eq(w['next_run'] > due['next_run'], True)

### Against the real shops

The same run, on a browser you are already logged into. `FossickCart` is the only line that changes. This is what having an interface bought.

```python
from ramabana import Agent
from ramabana.shop import FossickCart, cart_tools
from ramabana.vault import VaultHost

host  = VaultHost(roots=['~/notes'])              # the shared ~/.vishalakshi/vault.db
agent = Agent(host, model='gpt-mini', approvals=Approvals(policy=ask_before_writes))
agent.registry.tools.extend(cart_tools(FossickCart()))

agent.send('anything outstanding this week?')
```

Two things are worth doing before pointing it at a real trolley. Run it behind an `Approvals` policy: `cart_add` is in `WRITE_TOOLS`. Every add can be put in front of you. Leave the checkout alone. `fossick.shop` has `shop_fill` for the payment form and this deliberately does not wrap it. Filling a trolley is reversible. Placing the order is not.